# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys

%cd /content
!if ! [ -d "Deep-SVDD-PyTorch" ]; then git clone https://github.com/nicbk/Deep-SVDD-PyTorch.git; fi
%cd Deep-SVDD-PyTorch
!git checkout -b vae
!git pull origin vae
%cd ..
!mkdir -p "Deep-SVDD-PyTorch-DATA"

sys.path.append('/content/Deep-SVDD-PyTorch/src')
%cd Deep-SVDD-PyTorch-DATA

/content
Cloning into 'Deep-SVDD-PyTorch'...
remote: Enumerating objects: 1330, done.
remote: Counting objects: 100% (970/970), done.
remote: Compressing objects: 100% (274/274), done.
remote: Total 1330 (delta 765), reused 882 (delta 695), pack-reused 360
Receiving objects: 100% (1330/1330), 2.26 MiB | 1.41 MiB/s, done.
Resolving deltas: 100% (968/968), done.
/content/Deep-SVDD-PyTorch
Switched to a new branch 'vae'
From https://github.com/nicbk/Deep-SVDD-PyTorch
 * branch            vae        -> FETCH_HEAD
Updating 5c78641..365e6c9
Fast-forward
 src/datasets/celeba.py         |  22 +++++------
 src/networks/__init__.py       |   1 +
 src/networks/celeba_vae_Net.py | 125 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 src/networks/main.py           |  11 +++++-
 src/optim/ae_trainer.py        |  28 +++++++++-----
 src/optim/deepSVDD_trainer.py  |   6 ---
 6 files changed, 165 insertions(+), 28 deletions(-)
 create mode 100644 src/networks/celeba_vae_Net.py
/content
/co

# Train Model

In [3]:
import os
import click
import torch
import logging
import random
import numpy as np
import math
import matplotlib.pyplot as plt
import scipy
import copy

from PIL import Image
from utils.config import Config
from utils.visualization.plot_images_grid import plot_images_grid
from deepSVDD import DeepSVDD
from datasets.main import load_dataset

In [4]:
attributes = [
    '5_o_Clock_Shadow',
    'Arched_Eyebrows',
    'Attractive',
    'Bags_Under_Eyes',
    'Bald',
    'Bangs',
    'Big_Lips',
    'Big_Nose',
    'Black_Hair',
    'Blond_Hair',
    'Blurry',
    'Brown_Hair',
    'Bushy_Eyebrows',
    'Chubby',
    'Double_Chin',
    'Eyeglasses',
    'Goatee',
    'Gray_Hair',
    'Heavy_Makeup',
    'High_Cheekbones',
    'Male',
    'Mouth_Slightly_Open',
    'Mustache',
    'Narrow_Eyes',
    'No_Beard',
    'Oval_Face',
    'Pale_Skin',
    'Pointy_Nose',
    'Receding_Hairline',
    'Rosy_Cheeks',
    'Sideburns',
    'Smiling',
    'Straight_Hair',
    'Wavy_Hair',
    'Wearing_Earrings',
    'Wearing_Hat',
    'Wearing_Lipstick',
    'Wearing_Necklace',
    'Wearing_Necktie',
    'Young',
    'Gender: Man',
    'Gender: Woman',
    'Race: White',
    'Race: Black',
    'Race: Asian',
    'Race: Indian',
    'Race: Middle_Eastern',
    'Race: Latino',
    'Emotion: Angry',
    'Emotion: Disgust',
    'Emotion: Fear',
    'Emotion: Happy',
    'Emotion: Sad',
    'Emotion: Surprise',
    'Emotion: Neutral'
]

In [5]:
settings = {
    'dataset_name': 'celeba',
    'net_name': 'celeba_vae_Net',
    'xp_path': 'log/celeba_test_vae_smiling',
    'data_path': 'data',
    'train': False,
    'load_config': None,
    'load_model': 'log/celeba_test_vae_smiling/model.tar',
    'objective': 'one-class',
    'nu': 0.1,
    'device': 'cuda',
    'seed': -1,
    'optimizer_name': 'adam',
    'lr': 0.00002,
    'n_epochs': 34,
    'lr_milestone': [22],
    'batch_size': 128,
    'weight_decay': 0.5e-6,
    'pretrain': True,
    'ae_optimizer_name': 'adam',
    'ae_lr': 0.00002,
    'ae_n_epochs': 34,
    'ae_lr_milestone': [22],
    'ae_batch_size': 128,
    'ae_weight_decay': 0.5e-6,
    'n_jobs_dataloader': 0,
    'normal_class': 31
}

xp_path = settings['xp_path']
!mkdir -p $xp_path
data_path = settings['data_path']
!mkdir -p $data_path

Run the following box to import an existing model

In [8]:
!cp -r -v /content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling /content/Deep-SVDD-PyTorch-DATA/log/

'/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/log.txt' -> '/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/log.txt'
'/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/normals.png' -> '/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/normals.png'
'/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/outliers.png' -> '/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/outliers.png'
'/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/results.json' -> '/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/results.json'
'/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/model.tar' -> '/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/model.tar'
'/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/config.json' -> '/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/config.json'
'/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/freque

Only run the following box once

In [6]:
!cp -r -v /content/drive/MyDrive/CelebA/celeba /content/Deep-SVDD-PyTorch-DATA/data/celeba

%cd data/celeba/
!unzip img_align_celeba.zip
%cd ../../

Streaming output truncated to the last 5000 lines.
 extracting: img_align_celeba/197601.jpg  
 extracting: img_align_celeba/197602.jpg  
 extracting: img_align_celeba/197603.jpg  
 extracting: img_align_celeba/197604.jpg  
 extracting: img_align_celeba/197605.jpg  
 extracting: img_align_celeba/197606.jpg  
 extracting: img_align_celeba/197607.jpg  
 extracting: img_align_celeba/197608.jpg  
 extracting: img_align_celeba/197609.jpg  
 extracting: img_align_celeba/197610.jpg  
 extracting: img_align_celeba/197611.jpg  
 extracting: img_align_celeba/197612.jpg  
 extracting: img_align_celeba/197613.jpg  
 extracting: img_align_celeba/197614.jpg  
 extracting: img_align_celeba/197615.jpg  
 extracting: img_align_celeba/197616.jpg  
 extracting: img_align_celeba/197617.jpg  
 extracting: img_align_celeba/197618.jpg  
 extracting: img_align_celeba/197619.jpg  
 extracting: img_align_celeba/197620.jpg  
 extracting: img_align_celeba/197621.jpg  
 extracting: img_align_celeba/197622.jpg  
 ex

In [9]:
from numpy.core.numerictypes import obj2sctype
"""
Deep SVDD, a fully deep method for anomaly detection.

:arg DATASET_NAME: Name of the dataset to load.
:arg NET_NAME: Name of the neural network to use.
:arg XP_PATH: Export path for logging the experiment.
:arg DATA_PATH: Root path of data.
"""

# Get configuration
cfg = Config(settings.copy())

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
log_file = cfg.settings['xp_path'] + '/log.txt'
file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# If specified, load experiment config from JSON-file
if cfg.settings['load_config']:
    cfg.load_config(import_json=cfg.settings['load_config'])
    logger.info('Loaded configuration from %s.' % cfg.settings['load_config'])

# Print arguments
logger.info('Log file is %s.' % log_file)
logger.info('Data path is %s.' % cfg.settings['data_path'])
logger.info('Export path is %s.' % cfg.settings['xp_path'])

logger.info('Dataset: %s' % cfg.settings['dataset_name'])
logger.info('Normal class: %d' % cfg.settings['normal_class'])
logger.info('Network: %s' % cfg.settings['net_name'])

# Print configuration
logger.info('Deep SVDD objective: %s' % cfg.settings['objective'])
logger.info('Nu-paramerter: %.2f' % cfg.settings['nu'])

# Set seed
if cfg.settings['seed'] != -1:
    random.seed(cfg.settings['seed'])
    np.random.seed(cfg.settings['seed'])
    torch.manual_seed(cfg.settings['seed'])
    logger.info('Set seed to %d.' % cfg.settings['seed'])

# Default device to 'cpu' if cuda is not available
if not torch.cuda.is_available():
    cfg.settings['device'] = 'cpu'
logger.info('Computation device: %s' % cfg.settings['device'])
logger.info('Number of dataloader workers: %d' % cfg.settings['n_jobs_dataloader'])

# Load data
dataset = load_dataset(cfg.settings['dataset_name'], cfg.settings['data_path'], cfg.settings['normal_class'])
dataset_full = load_dataset(cfg.settings['dataset_name'], cfg.settings['data_path'], -1)

# Initialize DeepSVDD model and set neural network \phi
deep_SVDD = DeepSVDD(cfg.settings['objective'], cfg.settings['nu'])
deep_SVDD.set_network(cfg.settings['net_name'])
# If specified, load Deep SVDD model (radius R, center c, network weights, and possibly autoencoder weights)
if cfg.settings['load_model']:
    deep_SVDD.load_model(model_path=cfg.settings['load_model'], load_ae=True)
    logger.info('Loading model from %s.' % cfg.settings['load_model'])

logger.info('Pretraining: %s' % cfg.settings['pretrain'])
if cfg.settings['pretrain'] and cfg.settings['train']:
    # Log pretraining details
    logger.info('Pretraining optimizer: %s' % cfg.settings['ae_optimizer_name'])
    logger.info('Pretraining learning rate: %g' % cfg.settings['ae_lr'])
    logger.info('Pretraining epochs: %d' % cfg.settings['ae_n_epochs'])
    logger.info('Pretraining learning rate scheduler milestones: %s' % (cfg.settings['ae_lr_milestone'],))
    logger.info('Pretraining batch size: %d' % cfg.settings['ae_batch_size'])
    logger.info('Pretraining weight decay: %g' % cfg.settings['ae_weight_decay'])

    # Pretrain model on dataset (via autoencoder)
    deep_SVDD.pretrain(dataset_full,
                       optimizer_name=cfg.settings['ae_optimizer_name'],
                       lr=cfg.settings['ae_lr'],
                       n_epochs=cfg.settings['ae_n_epochs'],
                       lr_milestones=cfg.settings['ae_lr_milestone'],
                       batch_size=cfg.settings['ae_batch_size'],
                       weight_decay=cfg.settings['ae_weight_decay'],
                       device=cfg.settings['device'],
                       n_jobs_dataloader=cfg.settings['n_jobs_dataloader'])

# Train model on dataset
if cfg.settings['train']:
    # Log training details
    logger.info('Training optimizer: %s' % cfg.settings['optimizer_name'])
    logger.info('Training learning rate: %g' % cfg.settings['lr'])
    logger.info('Training epochs: %d' % cfg.settings['n_epochs'])
    logger.info('Training learning rate scheduler milestones: %s' % (cfg.settings['lr_milestone'],))
    logger.info('Training batch size: %d' % cfg.settings['batch_size'])
    logger.info('Training weight decay: %g' % cfg.settings['weight_decay'])
    deep_SVDD.train(dataset,
                    optimizer_name=cfg.settings['optimizer_name'],
                    lr=cfg.settings['lr'],
                    n_epochs=cfg.settings['n_epochs'],
                    lr_milestones=cfg.settings['lr_milestone'],
                    batch_size=cfg.settings['batch_size'],
                    weight_decay=cfg.settings['weight_decay'],
                    device=cfg.settings['device'],
                    n_jobs_dataloader=cfg.settings['n_jobs_dataloader'])

# Test model
deep_SVDD.test(dataset, device=cfg.settings['device'], n_jobs_dataloader=cfg.settings['n_jobs_dataloader'])

# Plot most anomalous and most normal (within-class) test samples
indices, labels, scores = zip(*deep_SVDD.results['test_scores'])
indices, labels, scores = np.array(indices), np.array(labels), np.array(scores)
idx_sorted = indices[labels == 0][np.argsort(scores[labels == 0])]  # sorted from lowest to highest anomaly score

# Return all attributes per image when test set is indexed
dataset.test_set.apply_target_transform = False

if cfg.settings['dataset_name'] in ('mnist', 'cifar10', 'celeba'):

    if cfg.settings['dataset_name'] == 'mnist':
        X_normals = dataset.test_set.data[idx_sorted[:32], ...].unsqueeze(1)
        X_outliers = dataset.test_set.data[idx_sorted[-32:], ...].unsqueeze(1)

    if cfg.settings['dataset_name'] == 'cifar10':
        X_normals = torch.tensor(np.transpose(dataset.test_set.data[idx_sorted[:32], ...], (0, 3, 1, 2)))
        X_outliers = torch.tensor(np.transpose(dataset.test_set.data[idx_sorted[-32:], ...], (0, 3, 1, 2)))

    if cfg.settings['dataset_name'] == 'celeba':
        X_normals = torch.from_numpy(np.transpose(np.array([np.array(Image.open(os.path.join(dataset.test_set.root, dataset.test_set.base_folder, "img_align_celeba", dataset.test_set.filename[idx_sorted[i]]))) for i in range(100)]), (0, 3, 1, 2)))
        X_outliers = torch.from_numpy(np.transpose(np.array([np.array(Image.open(os.path.join(dataset.test_set.root, dataset.test_set.base_folder, "img_align_celeba", dataset.test_set.filename[idx_sorted[-i]]))) for i in range(1, 101)]), (0, 3, 1, 2)))

        # Find frequency of tags in top 5% anomalous images
        label_frequency_outliers = [[attributes[i], 0] for i in range(55)]

        ilp_tags_outliers = []

        n_top_5p = math.floor(0.05 * len(idx_sorted))
        for i in idx_sorted[-1:-(n_top_5p + 1):-1].tolist():
            _, labels, _ = dataset.test_set[i]
            ilp_tags_outliers.append(labels.numpy())
            for j in range(55):
                if labels[j] == 1:
                    label_frequency_outliers[j][1] += 1

        frequency_sorted = copy.deepcopy(label_frequency_outliers)
        frequency_sorted.sort(key=lambda x: x[1])
        labels = [frequency_sorted[i][0] for i in range(55)]
        frequencies = [frequency_sorted[i][1] for i in range(55)]

        fig, ax = plt.subplots()
        ax.barh(labels, frequencies)
        ax.invert_yaxis()
        ax.set_xlabel('Frequency')
        ax.set_title('Frequency of attributes in top 5% most anomalous examples')
        fig.set_size_inches(13, 10)
        plt.savefig(cfg.settings['xp_path'] + '/frequency_outliers', dpi=300)
        plt.clf()

        # Find frequency of tags in top 95% normal images
        label_frequency_normals = [[attributes[i], 0] for i in range(55)]

        ilp_tags_normals = []

        n_bottom_95p = math.floor(0.95 * len(idx_sorted)) # Set to top 5% normal images for now
        for i in idx_sorted[:n_bottom_95p].tolist():
            _, labels, _ = dataset.test_set[i]
            ilp_tags_normals.append(labels.numpy())
            for j in range(55):
                if labels[j] == 1:
                    label_frequency_normals[j][1] += 1

        label_frequency = copy.deepcopy(label_frequency_normals)
        label_frequency.sort(key=lambda x : x[1])
        labels = [label_frequency[i][0] for i in range(55)]
        frequencies = [label_frequency[i][1] for i in range(55)]

        fig, ax = plt.subplots()
        ax.barh(labels, frequencies)
        ax.invert_yaxis()
        ax.set_xlabel('Frequency')
        ax.set_title('Frequency of attributes in top 95% most normal examples')
        fig.set_size_inches(13, 10)
        plt.savefig(cfg.settings['xp_path'] + '/frequency_normals', dpi=300)
        plt.clf()

    plot_images_grid(X_normals, export_img=cfg.settings['xp_path'] + '/normals', title='Most normal examples', padding=2)
    plot_images_grid(X_outliers, export_img=cfg.settings['xp_path'] + '/outliers', title='Most anomalous examples', padding=2)

# Save results, model, and configuration
if cfg.settings['train']:
    deep_SVDD.save_results(export_json=cfg.settings['xp_path'] + '/results.json')
    deep_SVDD.save_model(export_model=cfg.settings['xp_path'] + '/model.tar')
    cfg.save_config(export_json=cfg.settings['xp_path'] + '/config.json')

INFO:root:Log file is log/celeba_test_vae_smiling/log.txt.
INFO:root:Data path is data.
INFO:root:Export path is log/celeba_test_vae_smiling.
INFO:root:Dataset: celeba
INFO:root:Normal class: 31
INFO:root:Network: celeba_vae_Net
INFO:root:Deep SVDD objective: one-class
INFO:root:Nu-paramerter: 0.10
INFO:root:Computation device: cuda
INFO:root:Number of dataloader workers: 0
INFO:root:Loading model from log/celeba_test_vae_smiling/model.tar.
INFO:root:Pretraining: True
INFO:root:Starting testing...
INFO:root:Testing time: 43.765
INFO:root:Test set AUC: 57.15%
INFO:root:Finished testing.


IndexError: index 54 is out of bounds for dimension 0 with size 54

# Explanations

ILP - Minimise number of tags

In [ ]:
num_instances_outliers = n_top_5p
num_instances_normals = n_top_5p

# Parameters for ILP
c = [1 for i in range(55 * 2)] # 110 variables (The 2x55 matrix "X" must be flattened to a 1x80 matrix in order to match SciPy's problem format).
# Lower bound:
# Within a cluster, instances must map to at least `minimum_cover` tags.
# Across both clusters, tags must map to at least one instance.
minimum_cover = 3
b_l = [minimum_cover for i in range(num_instances_normals + num_instances_outliers)] + [1 for i in range(55)]
# Upper bound:
# Within a cluster, instances can map to arbitrarily many tags.
# Tags can map to both clusters at once
b_u = [np.inf for i in range(num_instances_normals + num_instances_outliers)] + [2 for i in range(55)]
bounds = scipy.optimize.Bounds(lb=0, ub=1) # All variables are binary
integrality = [1 for i in range(55 * 2)] # All variables are integer

merged_tags = [ilp_tags_normals[i].tolist() + [0 for j in range(55)] for i in range(num_instances_normals)]\
            + [[0 for j in range(55)] + ilp_tags_outliers[i].tolist() for i in range(num_instances_outliers)]\
            + [[1 if i == j else 0 for i in range(55)] + [1 if i == j else 0 for i in range(55)] for j in range(55)]

constraints = scipy.optimize.LinearConstraint(merged_tags, b_l, b_u)
sol = scipy.optimize.milp(c,
                          integrality=integrality,
                          bounds=bounds,
                          constraints=constraints)
if sol.status == 0:
    print('Solution found')
else:
    print('Solution not found')
    print('Error code: ' + str(sol.status))

print('')

normal_expl = sol.x[0:55]
outlier_expl = sol.x[55:]


# Print out explanation with tagging information

tag_coverage_normals = [0 for i in range(55)]
tag_coverage_normals_avg = []
tag_coverage_outliers = [0 for i in range(55)]
tag_coverage_outliers_avg = []

# Compute frequency of degree of instance nodes
for i in range(55):
    for k in range(num_instances_normals):
        tag = ilp_tags_normals[k]
        total_tags = 0
        for j in range(55):
            if normal_expl[j] == 1 and tag[j] == 1:
                total_tags += 1
        if total_tags == i + 1:
            tag_coverage_normals[i] += 1

    for k in range(num_instances_outliers):
        tag = ilp_tags_outliers[k]
        total_tags = 0
        for j in range(55):
            if outlier_expl[j] == 1 and tag[j] == 1:
                total_tags += 1
        if total_tags == i + 1:
            tag_coverage_outliers[i] += 1

print('Explanation of normal instances:')
print('Number of normal instances: ' + str(num_instances_normals))
for j in range(55):
        print('    Instances containing ' + str(j+1) + ' tag: ' + str(tag_coverage_normals[j]))
print('')

for i in range(55):
    in_cluster = 'yes' if normal_expl[i] == 1 else 'no '
    num_instances = sum(np.array(ilp_tags_normals)[:num_instances_normals,i])
    frequency_percent = "{:.2f}".format(100 * num_instances/num_instances_normals)
    initial_tabspace = '\t'
    if len(attributes[i]) < 15:
        initial_tabspace += '\t'
    if len(attributes[i]) < 7:
        initial_tabspace += '\t'

    print(attributes[i] + ':' + initial_tabspace + 'In_Cluster[' + in_cluster + '],\t' + 'Number:\t' + str(num_instances) + ',\t' + 'Frequency:\t' + frequency_percent + '%')

print('')

print('Explanation of outlier instances:')
print('Number of outlier instances: ' + str(num_instances_outliers))
for j in range(55):
        print('    Instances containing ' + str(j+1) + ' tag: ' + str(tag_coverage_outliers[j]))
print('')

for i in range(55):
    in_cluster = 'yes' if outlier_expl[i] == 1 else 'no '
    num_instances = sum(np.array(ilp_tags_outliers)[:np.array(num_instances_outliers),i])
    frequency_percent = "{:.2f}".format(100 * num_instances/num_instances_outliers)
    initial_tabspace = '\t'
    if len(attributes[i]) < 15:
        initial_tabspace += '\t'
    if len(attributes[i]) < 7:
        initial_tabspace += '\t'

    print(attributes[i] + ':' + initial_tabspace + 'In_Cluster[' + in_cluster + '],\t' + 'Number:\t' + str(num_instances) + ',\t' + 'Frequency:\t' + frequency_percent + '%')

verification_normals = idx_sorted[:21]
verification_outliers = idx_sorted[-1:-21:-1]

print('Verification of normal explanation')
for i in range(len(verification_normals)):
    _, label, i = dataset.test_set[verification_normals[i]]
    print('')
    print(i)
    cover_counter = 0
    for i in range(55):
        if label[i] == 1 and normal_expl[i] == 1:
            cover_counter += 1
    if cover_counter < minimum_cover:
        print('Formulation not working')
    else:
        print('Formulation working')

print('\n')
print('Verification of outlier explanation')
for i in range(len(verification_outliers)):
    _, label, i = dataset.test_set[verification_outliers[i]]
    print('')
    print(i)
    cover_counter = 0
    for i in range(55):
        if label[i] == 1 and outlier_expl[i] == 1:
            cover_counter += 1
    if cover_counter < minimum_cover:
        print('Formulation not working')
    else:
        print('Formulation working')


Solution found

Explanation of normal instances:
Number of normal instances: 499
    Instances containing 1 tag: 0
    Instances containing 2 tag: 3
    Instances containing 3 tag: 25
    Instances containing 4 tag: 47
    Instances containing 5 tag: 84
    Instances containing 6 tag: 127
    Instances containing 7 tag: 132
    Instances containing 8 tag: 62
    Instances containing 9 tag: 19
    Instances containing 10 tag: 0
    Instances containing 11 tag: 0
    Instances containing 12 tag: 0
    Instances containing 13 tag: 0
    Instances containing 14 tag: 0
    Instances containing 15 tag: 0
    Instances containing 16 tag: 0
    Instances containing 17 tag: 0
    Instances containing 18 tag: 0
    Instances containing 19 tag: 0
    Instances containing 20 tag: 0
    Instances containing 21 tag: 0
    Instances containing 22 tag: 0
    Instances containing 23 tag: 0
    Instances containing 24 tag: 0
    Instances containing 25 tag: 0
    Instances containing 26 tag: 0
    Insta

/usr/local/lib/python3.10/dist-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


ILP - Maximise number of edges

In [ ]:
num_instances_outliers = n_top_5p
num_instances_normals = n_top_5p

# Total number of edges in normal and outlier groups
num_edges_normals = sum([sum(instance) for instance in ilp_tags_normals[:num_instances_normals]])
num_edges_outliers = sum([sum(instance) for instance in ilp_tags_outliers[:num_instances_outliers]])

# Lists of tags per instance in normal and outlier groups
ilp_tags_normals_np = np.array(ilp_tags_normals)
ilp_tags_outliers_np = np.array(ilp_tags_outliers)

print('Percent of edges covered by each tag in normal group:')
for i in range(55):
    print(attributes[i] + ': ' + str(100*sum(ilp_tags_normals_np[:num_instances_normals,i])/num_edges_normals) + '%')

print('\n')

print('Percent of edges covered by each tag in outlier group:')
for i in range(55):
    print(attributes[i] + ': ' + str(100*sum(ilp_tags_outliers_np[:num_instances_outliers,i])/num_edges_outliers) + '%')

c = [-(sum(ilp_tags_normals_np[:num_instances_normals,i])/num_edges_normals
     - sum(ilp_tags_outliers_np[:num_instances_outliers,i])/num_edges_outliers)
      for i in range(55)]
  + [-(sum(ilp_tags_outliers_np[:num_instances_outliers,i])/num_edges_outliers
     - sum(ilp_tags_normals_np[:num_instances_normals,i])/num_edges_normals)
      for i in range(55)]

bounds = scipy.optimize.Bounds(lb=0, ub=1)
integrality = [1 for i in range(55 * 2)]

sol = scipy.optimize.milp(c,
                          integrality=integrality,
                          bounds=bounds)

print('')

if sol.status == 0:
    print('Solution found')
else:
    print('Solution not found')
    print('Error code: ' + str(sol.status))

print('')

normal_expl = sol.x[0:55]
outlier_expl = sol.x[55:]

# Remove tags that have similar frequency across both groups
for j in range(55):
    if 100*abs(sum(ilp_tags_normals_np[:num_instances_normals,j])/num_instances_normals
             - sum(ilp_tags_outliers_np[:num_instances_outliers,j])/num_instances_outliers) < 7:
        normal_expl[j] = 0
        outlier_expl[j] = 0

    # Reassign redundant "Gender: Male" tag to "Male" tag
    if j == 40:
        if normal_expl[40] == 1:
            normal_expl[20] = 1
            normal_expl[40] = 0
        else:
            normal_expl[20] = 0
        if outlier_expl[40] == 1:
            outlier_expl[20] = 1
            outlier_expl[40] = 0
        else:
            outlier_expl[20] = 0



# Print out explanation with tagging information

tag_coverage_normals = [0 for i in range(55)]
tag_coverage_normals_avg = []
tag_coverage_outliers = [0 for i in range(55)]
tag_coverage_outliers_avg = []

# Compute frequency of degree of instance nodes
for i in range(55):
    for k in range(num_instances_normals):
        tag = ilp_tags_normals_np[k]
        total_tags = 0
        for j in range(55):
            if normal_expl[j] == 1 and tag[j] == 1:
                total_tags += 1
        if total_tags == i + 1:
            tag_coverage_normals[i] += 1

    for k in range(num_instances_outliers):
        tag = ilp_tags_outliers_np[k]
        total_tags = 0
        for j in range(55):
            if outlier_expl[j] == 1 and tag[j] == 1:
                total_tags += 1
        if total_tags == i + 1:
            tag_coverage_outliers[i] += 1

print('Explanation of normal instances:')
print('Number of normal instances: ' + str(num_instances_normals))
for j in range(55):
        print('    Instances containing ' + str(j+1) + ' tag: ' + str(tag_coverage_normals[j]))
print('')

for i in range(55):
    in_cluster = 'yes' if normal_expl[i] == 1 else 'no '
    num_instances = sum(ilp_tags_normals_np[:num_instances_normals,i])
    frequency_percent = "{:.2f}".format(100 * num_instances/num_instances_normals)
    initial_tabspace = '\t'
    if len(attributes[i]) < 15:
        initial_tabspace += '\t'
    if len(attributes[i]) < 7:
        initial_tabspace += '\t'

    print(attributes[i] + ':' + initial_tabspace + 'In_Cluster[' + in_cluster + '],\t' + 'Number:\t' + str(num_instances) + ',\t' + 'Frequency:\t' + frequency_percent + '%')

print('')

print('Explanation of outlier instances:')
print('Number of outlier instances: ' + str(num_instances_outliers))
for j in range(55):
        print('    Instances containing ' + str(j+1) + ' tag: ' + str(tag_coverage_outliers[j]))
print('')

for i in range(55):
    in_cluster = 'yes' if outlier_expl[i] == 1 else 'no '
    num_instances = sum(ilp_tags_outliers_np[:num_instances_outliers,i])
    frequency_percent = "{:.2f}".format(100 * num_instances/num_instances_outliers)
    initial_tabspace = '\t'
    if len(attributes[i]) < 15:
        initial_tabspace += '\t'
    if len(attributes[i]) < 7:
        initial_tabspace += '\t'

    print(attributes[i] + ':' + initial_tabspace + 'In_Cluster[' + in_cluster + '],\t' + 'Number:\t' + str(num_instances) + ',\t' + 'Frequency:\t' + frequency_percent + '%')

print('')
print('Normal Explanation:')
print('')
for i in range(len(normal_expl)):
    if normal_expl[i] == 1 and attributes[i] != 'Smiling':
        print(attributes[i])

print('')
print('Outlier Explanation:')
print('')
for i in range(len(outlier_expl)):
    if outlier_expl[i] == 1 and attributes[i] != 'Smiling':
        print(attributes[i])

print('')
'''verification_normals = idx_sorted[:41]
verification_outliers = idx_sorted[-1:-41:-1]

print('Verification of normal explanation')
for i in range(len(verification_normals)):
    _, label, i = dataset.test_set[verification_normals[i]]
    print('')
    print(i)
    cover_counter = 0
    for i in range(55):
        if label[i] == 1 and normal_expl[i] == 1:
            cover_counter += 1
    print('Explanation covers: {} of tags'.format(cover_counter))

print('\n')
print('Verification of outlier explanation')
for i in range(len(verification_outliers)):
    _, label, i = dataset.test_set[verification_outliers[i]]
    print('')
    print(i)
    cover_counter = 0
    for i in range(55):
        if label[i] == 1 and outlier_expl[i] == 1:
            cover_counter += 1
    print('Explanation covers: {} of tags'.format(cover_counter))'''

Percent of edges covered by each tag in normal group:
5_o_Clock_Shadow: 0.35370684776457273%
Arched_Eyebrows: 3.1267685342388227%
Attractive: 4.385964912280702%
Bags_Under_Eyes: 1.400679117147708%
Bald: 0.19807583474816073%
Bangs: 1.245048104131296%
Big_Lips: 2.5608375778155064%
Big_Nose: 1.230899830220713%
Black_Hair: 0.1697792869269949%
Blond_Hair: 3.8907753254103%
Blurry: 0.3254102999434069%
Brown_Hair: 1.259196378041879%
Bushy_Eyebrows: 0.39615166949632147%
Chubby: 0.29711375212224106%
Double_Chin: 0.3678551216751556%
Eyeglasses: 0.1414827391058291%
Goatee: 0.1273344651952462%
Gray_Hair: 0.2829654782116582%
Heavy_Makeup: 4.032258064516129%
High_Cheekbones: 5.98471986417657%
Male: 1.5280135823429541%
Mouth_Slightly_Open: 5.362195812110922%
Mustache: 0.09903791737408037%
Narrow_Eyes: 1.3299377475947933%
No_Beard: 6.593095642331636%
Oval_Face: 2.77306168647425%
Pale_Skin: 0.22637238256932654%
Pointy_Nose: 2.6032823995472554%
Receding_Hairline: 0.8064516129032258%
Rosy_Cheeks: 1.103565

"verification_normals = idx_sorted[:41]\nverification_outliers = idx_sorted[-1:-41:-1]\n\nprint('Verification of normal explanation')\nfor i in range(len(verification_normals)):\n    _, label, i = dataset.test_set[verification_normals[i]]\n    print('')\n    print(i)\n    cover_counter = 0\n    for i in range(55):\n        if label[i] == 1 and normal_expl[i] == 1:\n            cover_counter += 1\n    print('Explanation covers: {} of tags'.format(cover_counter))\n\nprint('\n')\nprint('Verification of outlier explanation')\nfor i in range(len(verification_outliers)):\n    _, label, i = dataset.test_set[verification_outliers[i]]\n    print('')\n    print(i)\n    cover_counter = 0\n    for i in range(55):\n        if label[i] == 1 and outlier_expl[i] == 1:\n            cover_counter += 1\n    print('Explanation covers: {} of tags'.format(cover_counter))"

# Save Model

In [ ]:
!cp -rvf /content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling /content/drive/MyDrive/CelebA/celeba_log/

'/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling' -> '/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling'
'/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/frequency_outliers.png' -> '/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/frequency_outliers.png'
'/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/frequency_normals.png' -> '/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/frequency_normals.png'
'/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/log.txt' -> '/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/log.txt'
'/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/normals.png' -> '/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/normals.png'
'/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae_smiling/outliers.png' -> '/content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling/outliers.png'
'/content/Deep-SVDD-PyTorch-DATA/log/celeba_test_vae

# Global Contrast Normalization

The following box is only needed for computing Global Contrast Normalisation values. These are hardcoded into the corresponding \<model\>_Dataset class

In [ ]:
from torch.utils.data import Subset
from torchvision.datasets import CelebA
from base.torchvision_dataset import TorchvisionDataset
from datasets.preprocessing import get_target_label_idx, global_contrast_normalization
from datasets.celeba import MyCelebA
from datasets.cifar10 import MyCIFAR10
from typing import Any, Callable, List, Optional, Tuple, Union

import warnings
import torchvision.transforms as transforms


train_set_all = MyCelebA(root='data', split='all', target_type='attr', download=False, transform=None, target_transform=None)
pil_to_tensor = transforms.Compose([transforms.ToTensor()])

MIN = []
MAX = []
for normal_classes in range(0, 54):
    if normal_classes == 31:
        train_idx_normal = get_target_label_idx(train_set_full.attr[:,normal_classes].clone().data.cpu().numpy(), [1])
        train_set = Subset(train_set_full, train_idx_normal)
        print('Class: ' + str(normal_classes))

        _min_ = []
        _max_ = []

        for idx in train_set.indices:
            image = Image.open(os.path.join(train_set_full.root, train_set_full.base_folder, "img_align_celeba", train_set_full.filename[idx]))
            gcm = global_contrast_normalization(pil_to_tensor(image), 'l1')
            _min_.append(gcm.min())
            _max_.append(gcm.max())

        MIN.append(np.min(_min_))
        MAX.append(np.max(_max_))
        print(len(train_set.indices))
    else:
        MIN.append(0)
        MAX.append(0)
min_max = list(zip(MIN, MAX))
print(min_max)

# Embedding Similarity

Find similarity in embeddings between instances of tag

Since all samples are reparameterised to a scalar and drawn from a standard distribution with std=1, then it should make sense that the embedding variables most closely associated with the tag should have a lower magnitude of std deviation.

In [ ]:
import torchvision.utils as vutils
vae = deep_SVDD.ae_net

_, test_loader = dataset.loaders(batch_size=1)
test_load_i = iter(test_loader)

embeddings = []

for i in range(2500):
    x, _, index = next(test_load_i)

    x = vae.encoder(x)
    x = torch.flatten(x, start_dim=1)

    mu = vae.fc_mu(x)
    log_var = vae.fc_var(x)

    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)

    x = eps * std + mu

    embeddings.append(x)

embeddings = torch.stack([torch.tensor(embeddings[i]) for i in range(len(embeddings))])
embeddings_std = torch.std(embeddings, dim=(0, 1))
embeddings_mean = torch.mean(embeddings, dim=(0, 1))
for i in range(64):
    if embeddings_std[i] < 0.8:
        print('Similar variable: i=' + str(i) + ', std=' + str(embeddings_std[i]) + ', mean=' + str(embeddings_mean[i]))


/usr/local/lib/python3.10/dist-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


KeyboardInterrupt: ignored

# Generating new faces/counterfactuals

Generating faces for counterfactuals

In [ ]:
import torchvision.utils as vutils
vae = deep_SVDD.ae_net

def get_embeddings(x, index):
    vutils.save_image(x.data, 'face_original{}.png'.format(index), normalize=True)
    x = vae.encoder(x)
    x = torch.flatten(x, start_dim=1)

    mu = vae.fc_mu(x)
    log_var = vae.fc_var(x)

    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)

    x_e = eps * std + mu

    x = x_e.detach().clone()

    x = vae.decoder_input(x)
    x = x.view(-1, 768, 5, 5)
    x = vae.decoder(x)
    x = vae.final_layer(x)

    vutils.save_image(x, 'face_diff{}.png'.format(index), normalize=True)

    return x_e.detach().clone()

_, test_loader = dataset.loaders(batch_size=1)
test_load_i = iter(test_loader)

for i in range(2):
    x1, _, index1 = next(test_load_i)

for i in range(12):
    x2, _, index2 = next(test_load_i)

x1_e = get_embeddings(x1, index1[0])
x2_e = get_embeddings(x2, index2[0])

x3_e = torch.tensor([[0 for i in range(64)]], dtype=torch.float32)

for i in range(64):
    x3_e[0][i] = (x1_e[0][i] + x2_e[0][i]) / 2

x = vae.decoder_input(x3_e)
x = x.view(-1, 768, 5, 5)
x = vae.decoder(x)
x = vae.final_layer(x)

vutils.save_image(x, 'face_avg.png', normalize=True)

# Add Race & Gender tags to CelebA

In [ ]:
!pip install deepface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 11.9 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.5.0-py2.py3-none-any.whl size=116934 sha256=012010bdb1f6f6f1061a61705f8694041e9bb4f4dd3210fc82efb0e0b4394a21
  Stored in directory: /root/.cache/pip/wheels/90/d4/f7/9404e5db0116bd4d43e5666eaa3e70ab53723e1e3ea40c9a95
Successfully built fire


In [ ]:
from deepface import DeepFace
import csv
import multiprocessing

celeba_base = '/content/Deep-SVDD-PyTorch-DATA/data/celeba/'

def generate_csv_part(idx_sorted):
    raw_csv = open(celeba_base + 'list_attr_celeba.txt', 'r')
    csv_buf = raw_csv.readlines()
    raw_csv.close()

    attributes_csv = csv.reader(csv_buf,
                                delimiter=" ",
                                skipinitialspace=True)

    new_attributes = []

    new_attributes.append(next(attributes_csv))
    header = next(attributes_csv)
    genders = ['man', 'woman']
    races = ['white', 'black', 'asian', 'indian', 'middle_eastern', 'latino']
    emotions = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

    new_tags = genders + races + emotions

    if len(header) != 55:
        for tag in new_tags:
            header.append(tag)

    new_attributes.append(header)

    total_processed = 0
    successful_processed = 0

    for instance in attributes_csv:
        new_attributes.append(instance)

    for i in idx_sorted:
        index = int(dataset.test_set.filename[i][:-4]) + 1
        print(str(successful_processed)+'/'+str(total_processed) + ': ' + new_attributes[index][0])

        if len(new_attributes[index]) > 41:
            new_attributes[index] = new_attributes[index][:41]


        classes = {
            'dominant_gender': 'undefined',
            'dominant_race': 'undefined',
            'dominant_emotion': 'undefined'
        }

        try:
            classes_new = DeepFace.analyze(
                img_path=celeba_base + 'img_align_celeba/' + new_attributes[index][0],
                actions = ['gender', 'race', 'emotion']
            )[0]

            classes = classes_new

            successful_processed += 1
        except ValueError:
            pass

        classes['dominant_gender'] = classes['dominant_gender'].lower()
        if classes['dominant_race'] == 'middle eastern':
            classes['dominant_race'] = 'middle_eastern'
        if classes['dominant_race'] == 'latino hispanic':
            classes['dominant_race'] = 'latino'

        for gender in genders:
            if classes['dominant_gender'] == gender:
                new_attributes[index].append('1')
            else:
                new_attributes[index].append('-1')

        for race in races:
            if classes['dominant_race'] == race:
                new_attributes[index].append('1')
            else:
                new_attributes[index].append('-1')

        for emotion in emotions:
            if classes['dominant_emotion'] == emotion:
                new_attributes[index].append('1')
            else:
                new_attributes[index].append('-1')

        total_processed += 1

    print('Tagging success rate: ' + str(100*successful_processed/total_processed) + '%')

    return new_attributes

"""if __name__ == '__main__':
    with multiprocessing.Pool() as pool:
        manager = multiprocessing.Manager()
        lock = manager.Lock()

        data = [[i*10000, 1, lock] for i in range(19)]
        data += [[190000, 1, lock]]

        new_tags = []
        result = pool.starmap(generate_csv_part, data)
        for tag_block in result:
            new_tags += result"""

new_attributes = generate_csv_part(idx_sorted)
for instance in new_attributes[2:]:
    if len(instance) == 41:
        for i in range(15):
            instance.append(-1)

with open(celeba_base + 'list_attr_celeba_extra.txt', 'w') as output:
    writer = csv.writer(output,
                        delimiter=" ",
                        skipinitialspace=True)
    writer.writerows(new_attributes)


0/0: 188187.jpg


Action: emotion: 100%|██████████| 3/3 [00:00<00:00, 15.70it/s]


1/1: 193536.jpg


Action: emotion: 100%|██████████| 3/3 [00:00<00:00, 15.60it/s]


2/2: 191617.jpg


...

9562/9982: 200269.jpg
9562/9983: 188479.jpg
9562/9984: 185510.jpg
9562/9985: 194179.jpg


Action: emotion: 100%|██████████| 3/3 [00:00<00:00, 12.27it/s]


9563/9986: 188509.jpg


Action: emotion: 100%|██████████| 3/3 [00:00<00:00, 11.58it/s]


Tagging success rate: 95.7644938419946%


In [ ]:
!cp -r -v /content/Deep-SVDD-PyTorch-DATA/data/celeba/list_attr_celeba_extra.txt /content/drive/MyDrive/CelebA/celeba/list_attr_celeba_extra.txt

'/content/Deep-SVDD-PyTorch-DATA/data/celeba/list_attr_celeba_extra.txt' -> '/content/drive/MyDrive/CelebA/celeba/list_attr_celeba_extra.txt'


In [ ]:
import csv
celeba_base = '/content/Deep-SVDD-PyTorch-DATA/data/celeba/'

attributes_csv = csv.reader(open(celeba_base + 'list_attr_celeba.txt'),
                            delimiter=" ",
                            skipinitialspace=True)

new_attributes = []
for instance in attributes_csv:
    new_attributes.append(instance)

for instance in new_attributes:
    if len(instance) == 41:
        for i in range(15):
            instance.append(-1)

with open(celeba_base + 'list_attr_celeba_extra.txt', 'w') as output:
    writer = csv.writer(output,
                        delimiter=" ",
                        skipinitialspace=True)
    writer.writerows(new_attributes)